Pinn Architecture

In [ ]:
import torch
import torch.nn as nn

# --- 1. THE EXPERTS ---
class StraightExpert(nn.Module):
    """
    Specialist: Longitudinal Dynamics (Drag, Rolling, Engine Power).
    Blind to lateral forces.
    """
    def __init__(self):
        super().__init__()
        # Input: [Velocity, TireAge, TrackTemp, Compound_OneHot(3)] = 6 inputs
        self.net = nn.Sequential(
            nn.Linear(6, 32),
            nn.Tanh(),
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1), 
            nn.Softplus() # Friction must be positive
        )

    def forward(self, x):
        return self.net(x)

class CornerExpert(nn.Module):
    """
    Specialist: Lateral Dynamics (Slip Angle, Downforce, Centripetal).
    Obsessed with curvature.
    """
    def __init__(self):
        super().__init__()
        # Input: [Velocity, TireAge, TrackTemp, Compound_OneHot(3), Curvature] = 7 inputs
        self.net = nn.Sequential(
            nn.Linear(7, 64), # Bigger brain for complex physics
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 1), 
            nn.Softplus()
        )

    def forward(self, x_state, curvature):
        # Append curvature to the state vector
        # x_state shape: [Batch, 6], curvature: [Batch, 1]
        combined = torch.cat([x_state, curvature], dim=1)
        return self.net(combined)

# --- 2. THE BOSS (Mixture of Experts) ---
class MoEPINN(nn.Module):
    def __init__(self, blueprint):
        super().__init__()
        self.blueprint = blueprint
        
        # Physics Constants
        self.base_mass = blueprint['coefficients'].get('min_mass_kg', 798.0)
        
        # The Specialists
        self.straight_net = StraightExpert()
        self.corner_net = CornerExpert()
        
    def gate_function(self, curvature):
        """
        Returns weight 'w' (0.0 = Straight, 1.0 = Curve).
        Uses Sigmoid to switch smoothly when Radius < 500m (Curvature > 0.002).
        """
        return torch.sigmoid((torch.abs(curvature) - 0.002) * 1000)

    def forward(self, x_state, curvature):
        """
        Predicts Friction (Mu) by blending experts.
        """
        # 1. Ask both experts
        mu_straight = self.straight_net(x_state)
        mu_corner = self.corner_net(x_state, curvature)
        
        # 2. Decide who to trust
        w = self.gate_function(curvature)
        
        # 3. Blend
        return (1 - w) * mu_straight + (w) * mu_corner

    def simulate_strategy_batch(self, initial_state_batch, track_map, max_time=100.0):
        """
        Simulates MANY strategies (Batch) simultaneously with Adaptive Stepping.
        
        initial_state_batch: [Batch_Size, 7] -> [Vel, Age, Temp, C1, C2, C3, Distance]
        track_map: Tensor containing curvature for the whole lap.
        """
        batch_size = initial_state_batch.shape[0]
        device = initial_state_batch.device
        
        # Clone state to avoid modifying original
        current_state = initial_state_batch.clone()
        
        # Trackers
        distances = current_state[:, -1].clone() # Start at current distance (e.g. 2000m)
        times = torch.zeros(batch_size, device=device)
        
        # Mask: Who is still running?
        active_mask = torch.ones(batch_size, dtype=torch.bool, device=device)
        track_length = self.blueprint['track_length']
        
        # --- THE ADAPTIVE LOOP ---
        while active_mask.any():
            # A. Get Curvature (Vectorized Lookup)
            # Map current distance to track index
            cur_indices = (distances[active_mask].long() % int(track_length))
            # Safe clamp for index
            cur_indices = torch.clamp(cur_indices, 0, len(track_map)-1)
            
            curvatures = track_map[cur_indices].unsqueeze(1)
            
            # B. Decide Step Size (Adaptive)
            # If ANY active car is in a corner, slow down the whole batch for safety
            # (Optimization: You could mask sub-groups, but global min is safer for sync)
            max_k = torch.max(torch.abs(curvatures))
            if max_k > 0.005: 
                dt = 0.05 # Precision Mode
            else:
                dt = 0.20 # Speed Mode
                
            # C. PINN Physics Step
            active_states = current_state[active_mask]
            
            # Predict Mu (using MoE)
            # Pass state without distance column
            pred_mu = self.forward(active_states[:, :-1], curvatures) 
            
            # D. Apply Physics (Newton's 2nd Law)
            velocity = active_states[:, 0:1]
            mass = self.base_mass # Simplified constant mass for speed
            
            # Forces
            f_aero = 0.5 * 1.225 * 0.9 * 1.6 * (velocity**2)
            f_roll = 0.015 * mass * 9.81
            
            # Grip Limit
            downforce = 0.5 * 1.225 * 3.0 * 1.6 * (velocity**2)
            max_grip = pred_mu * (mass * 9.81 + downforce)
            
            # Lateral Demand
            f_lat = mass * (velocity**2) * torch.abs(curvatures)
            
            # Friction Circle: How much grip is left for accel?
            grip_sq = (max_grip**2) - (f_lat**2)
            available_long = torch.sqrt(torch.relu(grip_sq))
            
            # Throttle (Assume 100% for push lap simulation)
            engine_force = 15000.0 
            
            # Clamp force to available grip
            net_drive = torch.min(engine_force, available_long) - f_aero - f_roll
            acceleration = net_drive / mass
            
            # E. Integrate
            # v_new = v + a*dt
            new_velocity = velocity + (acceleration * dt)
            step_dist = new_velocity * dt
            
            # Update Tensors
            # We need to perform scattered updates to handle the mask
            # (Updating only the rows that are True in active_mask)
            
            # Update Velocity
            current_state[active_mask, 0] = new_velocity.squeeze()
            
            # Update Distance
            distances[active_mask] += step_dist.squeeze()
            current_state[active_mask, -1] = distances[active_mask]
            
            # Update Time
            times[active_mask] += dt
            
            # F. Check Lap Completion
            # If distance > target, stop
            # (Target is usually Start_Dist + Track_Len)
            # Here we just check if they covered 1 full lap distance worth
            target_dist = initial_state_batch[active_mask, -1] + track_length
            finished = distances[active_mask] >= target_dist
            
            # Update mask (Complex logic to turn off finished bits)
            # We find indices of currently active, then index into that...
            # For simplicity in this snippet, we just update active_mask
            # NOTE: Mask handling in loops is tricky. 
            # Simplified: Stop if time > max_time
            if times.max() > max_time:
                break
                
        return times

# --- 3. THE PHYSICS LOSS ENGINE ---
class RacingPhysicsLoss(nn.Module):
    def __init__(self, blueprint):
        super().__init__()
        # Load constants...
        self.base_mass = 798.0

    def forward(self, pred_mu, velocity, acceleration, curvature, throttle, brake, w_gate):
        """
        Calculates loss, split by Expert.
        """
        # 1. Common Force Calcs
        f_aero = 0.5 * 1.225 * 0.9 * 1.6 * (velocity**2)
        f_roll = 0.015 * 800 * 9.81
        f_lat_req = 800 * (velocity**2) * torch.abs(curvature)
        
        downforce = 0.5 * 1.225 * 3.0 * 1.6 * (velocity**2)
        max_grip = pred_mu * (800 * 9.81 + downforce)
        
        # 2. Straight Expert Loss (Longitudinal Focus)
        # F_net = ma -> Grip must cover (ma + Drag)
        f_long_req = (800 * acceleration) + f_aero + f_roll
        # Only penalize if we slip longitudinally
        loss_straight = torch.relu(torch.abs(f_long_req) - max_grip)
        
        # 3. Corner Expert Loss (Lateral Focus)
        # Grip must cover Centripetal Force
        loss_corner = torch.relu(f_lat_req - max_grip)
        
        # 4. Weighted Sum
        # If w=0 (Straight), we ignore corner loss.
        # If w=1 (Corner), we ignore straight loss (mostly).
        
        # Note: We actually want to enforce Friction Circle everywhere, 
        # but weighting helps the experts specialize.
        
        total_demand = torch.sqrt(f_lat_req**2 + f_long_req**2)
        loss_circle = torch.relu(total_demand - max_grip)
        
        return torch.mean(loss_circle**2)

Training

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

from pinn_model import MoEPINN, RacingPhysicsLoss

# CONFIGURATION
EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 0.001

def generate_physics_batch(blueprint, n_samples=2000):
    # ... (Same generation logic as before) ...
    # CRITICAL: Make sure we generate CURVATURE
    
    # 1. Random Distances
    track_len = blueprint['track_length']
    dist = np.random.uniform(0, track_len, n_samples)
    
    # 2. Look up Curvature (Simulated here, but use blueprint['maps'] in prod)
    # Let's fake a track with 3 corners
    curvature = np.sin(dist / 200.0) * 0.01 
    curvature[np.abs(curvature) < 0.003] = 0 # Make straights straighter
    
    # 3. Velocities (Slow in corners, Fast in straights)
    # Heuristic: V ~ 1 / sqrt(k)
    velocity = 30.0 + (50.0 * (1.0 - np.abs(curvature * 50)))
    velocity = np.clip(velocity, 20.0, 90.0)
    
    # Features: [Vel, Age, Temp, C1, C2, C3]
    X = np.zeros((n_samples, 6))
    X[:, 0] = velocity
    X[:, 1] = np.random.uniform(0, 20, n_samples) # Age
    X[:, 2] = 0.5 # Temp
    X[:, 5] = 1.0 # Hard Tire
    
    # Aux: [Curvature, Throttle, Brake]
    Aux = np.zeros((n_samples, 3))
    Aux[:, 0] = curvature
    # Throttle is high if curvature is low
    Aux[:, 1] = np.where(np.abs(curvature) < 0.002, 1.0, 0.0)
    
    return torch.tensor(X, dtype=torch.float32), torch.tensor(Aux, dtype=torch.float32)

def train_model():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using Device: {device}")
    
    # 1. Mock Blueprint
    blueprint = {'track_length': 5000, 'coefficients': {'min_mass_kg': 798}}
    
    # 2. Data
    X_train, Aux_train = generate_physics_batch(blueprint)
    X_train, Aux_train = X_train.to(device), Aux_train.to(device)
    
    dataset = TensorDataset(X_train, Aux_train)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    
    # 3. Model (MoE)
    model = MoEPINN(blueprint).to(device)
    physics_engine = RacingPhysicsLoss(blueprint).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 4. Loop
    print("Starting MoE Training...")
    loss_history = []
    
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        
        for batch_X, batch_Aux in loader:
            optimizer.zero_grad()
            
            # Physics Inputs
            vel = batch_X[:, 0:1]
            curv = batch_Aux[:, 0:1]
            thr = batch_Aux[:, 1:2]
            brk = batch_Aux[:, 2:3]
            
            # 1. Forward (MoE)
            pred_mu = model(batch_X, curv)
            
            # 2. Get Gating Weight (For logging or specialized loss)
            w = model.gate_function(curv)
            
            # 3. Fake Acceleration (Collocation placeholder)
            # If throttle is high, we expect accel.
            accel = (thr * 10.0) - (brk * 20.0)
            
            # 4. Loss
            loss = physics_engine(pred_mu, vel, accel, curv, thr, brk, w)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
        loss_history.append(total_loss)
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss {total_loss:.4f}")

    plt.plot(loss_history)
    plt.show()
    
    # Save
    torch.save(model.state_dict(), 'moe_pinn.pth')
    print("MoE Model Saved.")

if __name__ == "__main__":
    train_model()

Visualization

In [ ]:
import matplotlib.pyplot as plt

# Visualize the Training Loss
plt.figure(figsize=(10, 5))
plt.plot(loss_history) # You'll need to save loss values to a list in the loop
plt.title("PINN Physics Calibration")
plt.xlabel("Epochs")
plt.ylabel("Physics Loss")
plt.show()

# Visualize the Undercut Prediction
# (Assume 'gaps' is the output from the simulation loop)
plt.figure(figsize=(10, 5))
laps = [0, 1, 2, 3] # Outlap + 3 Push Laps
plt.plot(laps, gaps.detach().cpu().numpy()[0], marker='o', color='orange', label='Gap to Rival')
plt.axhline(0, color='green', linestyle='--', label='Overtake Point')
plt.title("Undercut Simulation: Norris vs Verstappen")
plt.xlabel("Laps into Stint")
plt.ylabel("Gap (seconds)")
plt.legend()
plt.grid(True)
plt.show()